In [3]:
from langchain_text_splitters import CharacterTextSplitter

sample_text ="청킹은 긴 문서를 작은 조각으로 나누는 과정입니다. 문맥 유지가 매우 중요합니다."

fixed_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=15,
    chunk_overlap=0
)

fixed_chunks = fixed_splitter.split_text(sample_text)

for i, chunk in enumerate(fixed_chunks):
    print(f"고정 청크 {i+1}: {chunk}")

고정 청크 1: 청킹은 긴 문서를 작은 조각
고정 청크 2: 으로 나누는 과정입니다. 문
고정 청크 3: 맥 유지가 매우 중요합니다.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

campus_rules = (
    "제1조 (목적)\n본 규정은 장학금 지급 기준을 정의합니다.\n\n"
    "제2조 (선발 기준)\n직전 학기 평점 3.5 이상이어야 하며 F학점이 없어야 합니다."
)

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=35,
    chunk_overlap=0
)

recursive_chunks = recursive_splitter.split_text(campus_rules)
for i, chunk in enumerate(recursive_chunks):
    print(f"재귀 청크 {i+1}: \n{chunk}\n")

재귀 청크 1: 
제1조 (목적)
본 규정은 장학금 지급 기준을 정의합니다.

재귀 청크 2: 
제2조 (선발 기준)

재귀 청크 3: 
직전 학기 평점 3.5 이상이어야 하며 F학점이 없어야

재귀 청크 4: 
합니다.



In [6]:
# 문맥이 끊어지기 쉬운 연속된 문장을 정의합니다.
notice_text = "기숙사 입사 신청은 8월 1일부터 시작됩니다. 제출 서류는 건강진단서이며 보건소에서 발급 가능합니다."

overlap_splitter = RecursiveCharacterTextSplitter(
    chunk_size=25,
    chunk_overlap=10
)

overlap_chunks = overlap_splitter.split_text(notice_text)

for i, chunk in enumerate(overlap_chunks):
    print(f"오버랩 청크 {i+1}: {chunk}")

오버랩 청크 1: 기숙사 입사 신청은 8월 1일부터 시작됩니다.
오버랩 청크 2: 시작됩니다. 제출 서류는 건강진단서이며
오버랩 청크 3: 건강진단서이며 보건소에서 발급 가능합니다.


In [10]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

model = init_chat_model(
    model_provider="openai",
    model="gpt-6-luna",
)

# 실습에 사용할 학사 공지 원문입니다.
raw_document = (
    "2026학년도 2학기 국가장학금 신청 안내입니다.\n"
    "신청 대상은 재학생 전체이며 직전 학기 평점 3.5 이상이어야 합니다.\n"
    "신청 마감일은 2026년 9월 30일 오후 6시까지입니다."
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

chunks = text_splitter.split_text(raw_document)

selected_chunk = chunks[1]

prompt = ChatPromptTemplate.from_messages([
    ("system", "제공된 [참고 문서]에만 근거하여 명확하게 답변하세요.\n\n[참고 문서]:\n{context}"),
    ("user","{question}")
])

rag_chain = prompt | model | StrOutputParser()

user_query = "장학금을 신청하기 위한 최소 학점 기준은 얼마인가요?"
response = rag_chain.invoke({
    "context": selected_chunk,
    "question": user_query
})

print("===[모델 답변]===")
print(response)

===[모델 답변]===
직전 학기 평점이 3.5 이상이어야 합니다.
